In [1]:
import sys
sys.path.insert(0, '..')

from helper.golbal_variables import * 
from modelpipeline import ModelPipeline

In [2]:
model_pipeline = ModelPipeline(
    path=GENRES_MEL,
    num_class=10,
    batch_size=64,
    epochs=40
)

model_pipeline.train(
    channel=1,
    linear_layer_out=512,
    rnn_hidden=512
)

Loading genre dataset
Classes   : {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}
Total imgs: 6000

Images per genre:
  blues          600 images
  classical      600 images
  country        600 images
  disco          600 images
  hiphop         600 images
  jazz           600 images
  metal          600 images
  pop            600 images
  reggae         600 images
  rock           600 images
[train] Dataset ready.

Classes: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}
Total images: 6000

Images per genre:
  blues         600 images
  classical     600 images
  country       600 images
  disco         600 images
  hiphop        600 images
  jazz          600 images
  metal         600 images
  pop           600 images
  reggae        600 images
  rock          600 images
Classes   : {'blues': 0, 'classical': 1, 'country': 2, 'dis

In [ ]:
from CRNN_model.pred import predict
from CRNN_model.crnn import CRNN

In [ ]:
genre_map = {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}

In [ ]:
import torch
import pandas as pd
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = CRNN(
    num_class=10,
    channel=1,
    height=400,
    width=600,
    linear_layer_out=256,
    rnn_hidden=256
)

checkpoint = torch.load("best_crnn_preload_images.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

In [ ]:
mashup_path = Path("Z:\\dlgen-AI-project\\data\\mashups_image")

In [ ]:
test_df = pd.read_csv(MESSY_DATA_DIR / "test.csv")
output = {}

for idx, row in test_df.iterrows():
    song_id = row["id"]
    filename = row["filename"]

    from pathlib import Path

    filename = Path(row["filename"]).stem
    image_path = mashup_path / f"{filename}.png"
    result = predict(
        model=model,
        image_path=image_path,
        device=device,
        genre_map=genre_map
    )

    output[song_id] = result
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(test_df)} songs...")

output_df = pd.DataFrame(list(output.items()), columns=['id', 'genre'])
output_df = output_df.set_index('id')
output_df.to_csv("output.csv")